In [ ]:
import pandas as pd
DIR1 = r'../data/clean/labeled_data.csv'
DIR2 = r'../data/clean/standarized.csv'
data = pd.read_csv(DIR1, index_col = 0)
st_data = pd.read_csv(DIR2, index_col = 0)
print(data[['label', 'sektor']].sort_values(by = ['label', 'sektor']))

In [ ]:
melted = pd.melt(st_data, 
                 id_vars = ['label', 'sektor'], 
                 value_vars = ['ROA', 'AAGR', 'DY', 'P/E', 'beta'], 
                 var_name = 'zmienna',
                 ignore_index = False)
print(melted.head())

In [ ]:
import seaborn as sns
from src.processing import set_plot_style, set_plot_font, save_chart
import matplotlib.pyplot as plt
set_plot_style()
set_plot_font()
sns.catplot(data = data, x = 'label', hue = 'sektor', kind = 'count', palette = 'grey')
plt.xlabel('')
plt.ylabel('Liczebność')
save_chart('struktura')

In [ ]:
import numpy as np
sns.catplot(data=melted, 
            kind='point', 
            x='zmienna', 
            y='value', 
            hue='label',
            errorbar=None,
            palette='Greys_d',
            linestyles=['--', '-.', ':'],
            markers=['o', 'v', 's'],
            height=5, 
            aspect=1.3
            )
plt.xlabel('')
plt.ylabel('Wartość')
plt.ylim((-1,2))
plt.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.3)
save_chart('means')

In [ ]:
from sklearn.metrics import silhouette_score
from src.model_implementation import create_pipeline

X = data[['ROA', 'AAGR', 'DY', 'P/E', 'beta']]
labels = data['label']
labels_c = labels.map({'A':0,'B':1,'C':2})
pipeline = create_pipeline(clusters=3)
pipeline.fit(X)
X_transformed = pipeline[:-1].transform(X)
silhouette = silhouette_score(X_transformed, labels=labels)

print(silhouette)

In [ ]:
from sklearn.decomposition import PCA

pca_3dim = PCA(n_components=3).fit(X_transformed)
data_3dim_pca = pca_3dim.transform(X_transformed)
data_pca_2dim = data_3dim_pca[:, [0, 1]]
print(pca_3dim.explained_variance_ratio_)

In [ ]:
sns.heatmap(pd.DataFrame(X_transformed).corr(), vmin=-1, vmax=1, annot=True)

In [ ]:
from src.processing import set_plot_font, save_chart
import matplotlib.pyplot as plt
import numpy as np
pca_data = pd.DataFrame(data_3dim_pca, columns=['x','y', 'z'], index=labels.index)
pca_data['label']=labels.values
markers=['v', 'o', 'D']
lab = ['A', 'B', 'C']


In [ ]:
%matplotlib widget
fig = plt.figure(figsize=(12,6))
fig.patch.set_facecolor('white')
fig.subplots_adjust(wspace=0.1, left=0.05, right=0.95)

with plt.style.context(['grayscale']):
    ax1 = fig.add_subplot(1, 2, 1)
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    for l, m in zip(lab, markers):
        pairs = pca_data[pca_data['label']==l]
        ax1.scatter(pairs['x'], pairs['y'], marker=m, label=l, s=50, linewidth=0.5, edgecolors='black')
        ax2.scatter(pairs['x'], pairs['y'], pairs['z'],marker=m, label=l, s=100, linewidth=0.5, edgecolors='black', alpha=.8)
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=3, frameon=True)
    ax1.grid(None)
    ax1.set_title('Projekcja dwuwymiarowa - 56% wariancji')
    ax2.set_title('Projekcja trójwymiarowa - 77% wariancji')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_yticks(np.arange(-2, 3, 1))
    ax2.set_zticks(np.arange(-2, 3, 1))
    ax2.view_init(elev=20, azim=-60)
    ax2.xaxis.pane.fill = False
    ax2.yaxis.pane.fill = False
    ax2.zaxis.pane.fill = False
    pos2 = ax2.get_position()

save_chart('3d_2d')